In [9]:
import logging as log
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import ValenceType
from sklearn.cross_validation import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn

ATOM_SDF_FILENAME = "Conformer3D_COMPOUND_CID_4(1).sdf"

ATOM_FEATURE_COLUMNS = [
    "bondCount",
    "totalHydrogenCount",
    "valency",
    "mass",
    "hybridizationEncoded",
    "isAromaticInt",
    "charge",
]

RANDOM_SEED = 42


def get_data_dir():
    data_directory = os.environ["DATA_DIR"]
    log.info(f"Data directory: {data_directory}")
    return data_directory


def resolve_data_path(filename: str) -> Path:
    return Path(get_data_dir()) / filename


@dataclass(frozen=True)
class PreparedDataset:
    name: str
    task_type: str
    frame: pd.DataFrame
    feature_columns: list[str]
    target_column: str | None
    description: str
    class_names: list[str] | None = None

    def feature_matrix(self) -> np.ndarray:
        return self.frame.loc[:, self.feature_columns].to_numpy(dtype=np.float64)

    def target_vector(self) -> np.ndarray:
        if self.target_column is None:
            raise ValueError(f"Dataset {self.name} does not define a target column")

        return self.frame[self.target_column].to_numpy()

    def summary(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "task_type": self.task_type,
            "row_count": int(len(self.frame)),
            "feature_count": int(len(self.feature_columns)),
            "feature_columns": list(self.feature_columns),
            "target_column": self.target_column,
            "class_names": None if self.class_names is None else list(self.class_names),
            "description": self.description,
        }


@dataclass(frozen=True)
class SupervisedSplit:
    x_train: np.ndarray
    x_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray
    scaler: StandardScaler | None
    used_holdout_split: bool
    evaluation_note: str


def load_sdf_molecules(filename: str) -> list[Chem.Mol]:
    return get_valid_molecules(str(resolve_data_path(filename)))


def get_valid_molecules(sdf_file_path: str) -> list[Chem.Mol]:
    with Chem.SDMolSupplier(sdf_file_path, removeHs=False) as supplier:
        return [mol for mol in supplier if mol is not None]


def extract_atom_feature_matrix(molecules: list[Chem.Mol]) -> pd.DataFrame:
    atom_data: list[dict[str, object]] = []

    for molecule_index, mol in enumerate(molecules):
        for atom in mol.GetAtoms():
            atom_data.append(
                {
                    "moleculeIndex": molecule_index,
                    "index": atom.GetIdx(),
                    "bondCount": atom.GetDegree(),
                    "charge": atom.GetFormalCharge(),
                    "implicitHydrogenCount": atom.GetNumImplicitHs(),
                    "totalHydrogenCount": atom.GetTotalNumHs(),
                    "atomicNumber": atom.GetAtomicNum(),
                    "symbol": atom.GetSymbol(),
                    "valency": atom.GetValence(which=ValenceType.EXPLICIT),
                    "isAromatic": atom.GetIsAromatic(),
                    "mass": atom.GetMass(),
                    "hybridization": str(atom.GetHybridization()),
                    "properties": atom.GetPropsAsDict(),
                }
            )

    atom_feature_df = pd.DataFrame(atom_data)
    if atom_feature_df.empty:
        raise ValueError("Expected at least one atom when building the atom feature matrix")

    return atom_feature_df


def encode_categories(series: pd.Series) -> pd.Series:
    values = series.astype("string").str.strip().fillna("Unknown")
    values = values.mask(values.eq(""), "Unknown")
    category_order = sorted(values.unique().tolist())
    mapping = {value: index for index, value in enumerate(category_order)}
    return values.map(mapping).astype(int)


def build_atom_feature_frame(
    filename: str = ATOM_SDF_FILENAME,
) -> tuple[pd.DataFrame, Chem.Mol]:
    molecules = load_sdf_molecules(filename)

    if not molecules:
        raise ValueError(f"No valid molecules were loaded from {filename}")

    molecule = molecules[0]
    atom_feature_df = extract_atom_feature_matrix([molecule]).copy()
    atom_feature_df["atomId"] = atom_feature_df["index"].astype(int) + 1
    atom_feature_df["hybridizationEncoded"] = encode_categories(atom_feature_df["hybridization"])
    atom_feature_df["isAromaticInt"] = atom_feature_df["isAromatic"].astype(int)
    atom_feature_df["isHeavyAtom"] = atom_feature_df["atomicNumber"].astype(int).gt(1).astype(int)
    atom_feature_df["elementLabel"] = atom_feature_df["symbol"].astype(str)
    atom_feature_df["atomLabel"] = atom_feature_df["symbol"].astype(str) + atom_feature_df["atomId"].astype(str)
    return atom_feature_df, molecule


def build_atom_element_dataset(filename: str = ATOM_SDF_FILENAME) -> PreparedDataset:
    atom_feature_df, _ = build_atom_feature_frame(filename)
    class_names = sorted(atom_feature_df["elementLabel"].astype(str).unique().tolist())
    label_to_index = {label: index for index, label in enumerate(class_names)}
    frame = atom_feature_df.loc[:, ["atomLabel", *ATOM_FEATURE_COLUMNS]].copy()
    frame["elementTarget"] = atom_feature_df["elementLabel"].map(label_to_index).astype(int)

    return PreparedDataset(
        name="atom-element-multiclass",
        task_type="classification",
        frame=frame,
        feature_columns=ATOM_FEATURE_COLUMNS,
        target_column="elementTarget",
        class_names=class_names,
        description="Atom-level multiclass classification over O/N/C/H using non-symbol features.",
    )


def can_build_holdout_split(task_type: str, y_values: np.ndarray) -> bool:
    if len(y_values) < 8:
        return False

    if task_type != "classification":
        return True

    _, counts = np.unique(y_values, return_counts=True)
    return len(counts) > 1 and int(counts.min()) >= 2


def build_supervised_split(
    dataset: PreparedDataset,
    *,
    test_size: float = 0.3,
    random_state: int = RANDOM_SEED,
    scale_features: bool = True,
) -> SupervisedSplit:
    x_values = dataset.feature_matrix()
    y_values = dataset.target_vector()
    used_holdout_split = can_build_holdout_split(dataset.task_type, y_values)

    if used_holdout_split:
        stratify = y_values if dataset.task_type == "classification" else None
        x_train, x_test, y_train, y_test = train_test_split(
            x_values,
            y_values,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )
        evaluation_note = "Evaluation uses a deterministic holdout split."
    else:
        x_train = x_values
        x_test = x_values
        y_train = y_values
        y_test = y_values
        evaluation_note = (
            "Dataset is too small for a stable holdout split, so evaluation is reported on the training rows."
        )

    scaler = None
    if scale_features:
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_test = scaler.transform(x_test)

    return SupervisedSplit(
        x_train=x_train,
        x_test=x_test,
        y_train=y_train,
        y_test=y_test,
        scaler=scaler,
        used_holdout_split=used_holdout_split,
        evaluation_note=evaluation_note,
    )


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str] | None) -> dict[str, Any]:
    labels = None if class_names is None else list(range(len(class_names)))
    confusion = confusion_matrix(y_true, y_pred, labels=labels)

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": confusion.astype(int).tolist(),
        "class_names": class_names,
    }


def run_torch_classification(dataset: PreparedDataset, epochs: int = 200) -> dict[str, Any]:
    if int(np.unique(dataset.target_vector()).size) < 2:
        return {
            "status": "insufficient_data",
            "library": "pytorch",
            "dataset": dataset.summary(),
            "reason": "PyTorch classification requires at least two target classes.",
        }

    split = build_supervised_split(dataset)
    x_train = torch.tensor(split.x_train, dtype=torch.float32)
    y_train = torch.tensor(split.y_train.astype(np.int64), dtype=torch.long)
    x_eval = torch.tensor(split.x_test, dtype=torch.float32)
    output_dim = len(dataset.class_names or np.unique(split.y_train))
    model = nn.Sequential(
        nn.Linear(x_train.shape[1], 16),
        nn.ReLU(),
        nn.Linear(16, output_dim),
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        predictions = model(x_eval).argmax(dim=1).cpu().numpy()

    return {
        "dataset": dataset.summary(),
        "evaluation_note": split.evaluation_note,
        "epochs": int(epochs),
        "metrics": classification_metrics(split.y_test, predictions, dataset.class_names),
    }

In [10]:
atom_element_dataset = build_atom_element_dataset()
run_torch_classification(atom_element_dataset)

{'library': 'pytorch',
 'dataset': {'name': 'atom-element-multiclass',
  'task_type': 'classification',
  'row_count': 14,
  'feature_count': 7,
  'feature_columns': ['bondCount',
   'totalHydrogenCount',
   'valency',
   'mass',
   'hybridizationEncoded',
   'isAromaticInt',
   'charge'],
  'target_column': 'elementTarget',
  'class_names': ['C', 'H', 'N', 'O'],
  'description': 'Atom-level multiclass classification over O/N/C/H using non-symbol features.'},
 'evaluation_note': 'Dataset is too small for a stable holdout split, so evaluation is reported on the training rows.',
 'epochs': 200,
 'metrics': {'accuracy': 1.0,
  'macro_f1': 1.0,
  'confusion_matrix': [[3, 0, 0, 0], [0, 9, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]],
  'class_names': ['C', 'H', 'N', 'O']}}